# ЛР2. Анализ результатов экспериментов

Блокнот **только читает** результаты: обучение и оценивание выполняются скриптами
`python -m src.train --all` и `python -m src.evaluate`. Скрытой логики обучения здесь нет.
Перед запуском выполните эти две команды из корня проекта.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)

## 1. Сводная таблица E0–E4 (CV внутри train, тест — один раз)

In [ ]:
res = pd.read_csv("reports/lab02-results.csv")
cols = ["id", "best_params", "cv_mse_mean", "cv_mse_std", "cv_mae_mean", "cv_r2_mean",
        "test_mse", "test_mae", "test_r2", "test_n_negative_predictions", "selected"]
res[cols].sort_values("cv_mse_mean").round(3)

Выбор итоговой модели сделан **только по CV** по правилу из `configs/base.yaml`.
Обратите внимание на std: разница между лучшими лог-моделями (E1_esharma3, E1_ols_loglog,
E2, E4_ridge_loglog_vproxy) меньше одного стандартного отклонения.

## 2. Мультиколлинеарность: VIF и корреляции

In [ ]:
display(pd.read_csv("reports/tables/lab02-vif.csv", index_col=0))
pd.read_csv("reports/tables/lab02-correlations.csv", index_col=0)

## 3. Устойчивость коэффициентов по 15 разбиениям CV

In [ ]:
for run in ["e1_ols", "e1_ols_loglog", "e2_ridge_loglog", "e4_ols_vproxy", "e4_ridge_vproxy"]:
    c = pd.read_csv(f"runs/{run}/coefficients.csv", index_col=0)
    print(run); display(c[c.index.str.startswith("num__")].round(3))

## 4. Остатки

In [ ]:
for f in ["lab02-e1_ols-residuals-oof.png", "lab02-e1_esharma3-residuals-oof.png",
          "lab02-e1_esharma3-pred-vs-true.png"]:
    display(Image(f"reports/figures/{f}"))

## 5. Анализ ошибок итоговой модели

In [ ]:
run = res.loc[res["selected"], "run_name"].item()
for key in ["zones_oof", "top_errors_oof", "top_errors_test", "mae_by_species_oof",
            "mae_by_weight_oof", "mae_by_weight_test"]:
    print(key); display(pd.read_csv(f"reports/tables/lab02-{run}-{key}.csv"))

## 6. Отрицательные прогнозы

In [ ]:
neg = pd.read_csv("reports/tables/lab02-negative-predictions.csv")
neg[neg["n_negative"] > 0]